# 可转债数据采集 重构

## 元数据
- 原始路径: g:\我的云端硬盘\workspace\projects\github\work_station\代码库\项目\可转债数据
- 创建时间: 2026-02-15
- 任务ID: T24

## 1. 项目概述

### 1.1 功能描述
本项目用于采集和管理可转债市场数据，主要功能包括：
- 从同花顺(iFinD)数据接口获取可转债行情数据
- 支持历史数据采集和数据补充
- 将数据存储到MySQL数据库
- 提供交易日历管理功能

### 1.2 数据源
- **同花顺iFinD数据接口**: 通过iFinDPy库获取可转债市场数据
- **数据库**: bond.marketinfo_equity、bond.basicinfo_equity

### 1.3 输出结果
- 数据库表: marketinfo_equity1 (可转债行情数据)
- 采集指标: 收盘价、成交额、纯债溢价率、纯债到期收益率、转股溢价率、转股市盈率、正股市盈率、正股市净率、转股市净率、未转股余额

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import sys
import json
import time
import logging
import traceback
from datetime import datetime, timedelta
from typing import List, Dict, Optional, Tuple

# 数据处理
import pandas as pd
import numpy as np

# 数据库
from sqlalchemy import create_engine, text

# 同花顺数据接口 (需要安装iFinDPy)
try:
    from iFinDPy import THS_iFinDLogin, THS_iFinDLogout, THS_DS
    THS_AVAILABLE = True
except ImportError:
    THS_AVAILABLE = False
    print("警告: iFinDPy未安装，同花顺数据接口不可用")

# 导入配置
from config import *

print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

### 2.2 配置参数

In [ ]:
# 数据采集配置
BATCH_SIZE = 100  # 批量处理大小
REQUEST_DELAY = 0.2  # 请求间隔(秒)
MAX_RETRY = 3  # 最大重试次数

# 交易日历缓存文件
TRADING_DAYS_CACHE_FILE = DATA_DIR / 'trading_days_cache_a.json'

# 指标配置
INDICATORS_CONFIG = [
    {'name': 'ths_bond_close_cbond', 'param': '103', 'rename': 'close', 'desc': '收盘价'},
    {'name': 'ths_new_bond_amt_cbond', 'param': '', 'rename': 'amount', 'desc': '成交额'},
    {'name': 'ths_pure_bond_premium_rate_cbond', 'param': '', 'rename': 'pure_premium', 'desc': '纯债溢价率'},
    {'name': 'ths_pure_bond_ytm_cbond', 'param': '', 'rename': 'ytm', 'desc': '纯债到期收益率'},
    {'name': 'ths_conversion_premium_rate_cbond', 'param': '', 'rename': 'conv_premium', 'desc': '转股溢价率'},
    {'name': 'ths_conver_pe_cbond', 'param': '', 'rename': 'conv_pe', 'desc': '转股市盈率'},
    {'name': 'ths_stock_pe_cbond', 'param': '', 'rename': 'stock_pe', 'desc': '正股市盈率'},
    {'name': 'ths_stock_pb_cbond', 'param': '', 'rename': 'stock_pb', 'desc': '正股市净率'},
    {'name': 'ths_conver_pb_cbond', 'param': '', 'rename': 'conv_pb', 'desc': '转股市净率'},
    {'name': 'ths_un_conversion_balance_cbond', 'param': '', 'rename': 'un_conversion_balance', 'desc': '未转股余额'}
]

print("配置参数加载完成")

## 3. 数据获取

### 3.1 数据库连接

In [ ]:
def create_db_engine():
    """创建数据库引擎"""
    conn_str = (
        f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}"
        f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )
    return create_engine(conn_str)

# 创建数据库引擎
engine = create_db_engine()
print("数据库连接创建成功")

### 3.2 同花顺数据接口连接

In [ ]:
def connect_ths(username: str, password: str) -> bool:
    """连接同花顺数据接口
    
    Args:
        username: 同花顺账号
        password: 同花顺密码
        
    Returns:
        bool: 连接是否成功
    """
    if not THS_AVAILABLE:
        print("同花顺数据接口不可用")
        return False
    
    retry_count = 0
    while retry_count < MAX_RETRY:
        login_result = THS_iFinDLogin(username, password)
        if login_result == 0:
            print("同花顺数据接口连接成功")
            return True
        else:
            retry_count += 1
            print(f"连接失败，错误码：{login_result}，第{retry_count}次重试")
            time.sleep(5)
    
    print("同花顺数据接口连接失败")
    return False

def disconnect_ths():
    """断开同花顺数据接口"""
    if THS_AVAILABLE:
        THS_iFinDLogout()
        print("同花顺数据接口已断开")

### 3.3 获取可转债代码列表

In [ ]:
def get_bond_codes(engine) -> pd.DataFrame:
    """获取需要采集的可转债代码
    
    Returns:
        pd.DataFrame: 包含 trade_code、delist_date 和 list_date 的 DataFrame
    """
    query = """
        SELECT DISTINCT A.trade_code,
        B.ths_delist_date_bond as delist_date,
        B.ths_listed_date_bond as list_date
        FROM bond.marketinfo_equity A
        INNER JOIN bond.basicinfo_equity B
        ON A.trade_code=B.trade_code
        WHERE A.ths_bond_balance_bond>0
        AND A.trade_code NOT LIKE 'S%%'
        AND A.trade_code NOT LIKE '%%NQ'
        AND A.trade_code NOT LIKE '%%.00'
        AND A.trade_code NOT LIKE '%%ZJEE'
    """
    
    with engine.connect() as conn:
        df = pd.read_sql(query, conn)
    
    print(f"获取到 {len(df)} 个可转债代码")
    return df

# 测试获取代码
bond_codes_df = get_bond_codes(engine)
bond_codes_df.head()

### 3.4 加载交易日历

In [ ]:
def load_trading_days(cache_file: str = None) -> List[str]:
    """加载交易日历
    
    Args:
        cache_file: 交易日历缓存文件路径
        
    Returns:
        List[str]: 交易日列表
    """
    if cache_file is None:
        cache_file = str(TRADING_DAYS_CACHE_FILE)
    
    try:
        with open(cache_file, 'r', encoding='utf-8') as f:
            cache_data = json.load(f)
            trading_days = cache_data['trading_days']
        print(f"已加载 {len(trading_days)} 个交易日")
        return trading_days
    except Exception as e:
        print(f"加载交易日历失败: {str(e)}")
        return []

# 加载交易日历
trading_days = load_trading_days()
print(f"交易日历范围: {trading_days[0]} 至 {trading_days[-1]}")

## 4. 数据处理

### 4.1 数据采集函数

In [ ]:
def collect_bond_data(bond_codes: List[str], start_date: str, end_date: str) -> Optional[pd.DataFrame]:
    """采集可转债数据
    
    Args:
        bond_codes: 可转债代码列表
        start_date: 开始日期 (YYYY-MM-DD)
        end_date: 结束日期 (YYYY-MM-DD)
        
    Returns:
        pd.DataFrame: 采集的数据
    """
    if not THS_AVAILABLE:
        print("同花顺数据接口不可用，无法采集数据")
        return None
    
    codes_str = ','.join(bond_codes)
    all_data = []
    
    # 逐个获取指标数据
    for indicator in INDICATORS_CONFIG:
        result = THS_DS(
            codes_str,
            indicator['name'],
            indicator['param'],
            '',
            start_date,
            end_date
        )
        
        if result.data is None:
            print(f"获取{indicator['desc']}数据失败: {result.errmsg}")
            continue
            
        df = result.data
        if not df.empty:
            all_data.append(df)
        
        # 避免请求过于频繁
        time.sleep(REQUEST_DELAY)
    
    if not all_data:
        print(f"未获取到任何数据: {codes_str}")
        return None
    
    # 合并所有数据
    final_df = all_data[0]
    for df in all_data[1:]:
        final_df = pd.merge(final_df, df, on=['time', 'thscode'], how='outer')
    
    # 重命名列
    rename_dict = {'time': 'dt', 'thscode': 'trade_code'}
    for indicator in INDICATORS_CONFIG:
        rename_dict[indicator['name']] = indicator['rename']
    
    final_df = final_df.rename(columns=rename_dict)
    
    return final_df

### 4.2 数据清洗与转换

In [ ]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """清洗数据
    
    Args:
        df: 原始数据
        
    Returns:
        pd.DataFrame: 清洗后的数据
    """
    # 确保日期列是字符串类型 YYYY-MM-DD
    df['dt'] = pd.to_datetime(df['dt']).dt.strftime('%Y-%m-%d')
    
    # 确保trade_code是字符串类型
    df['trade_code'] = df['trade_code'].astype(str)
    
    # 转换数值列，处理无效数据
    numeric_cols = [ind['rename'] for ind in INDICATORS_CONFIG]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            # 将特殊值（inf, -inf）转换为NaN
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    
    return df

## 5. 核心逻辑

### 5.1 主函数实现

In [ ]:
class BondDataCollector:
    """可转债数据采集器"""
    
    def __init__(self, ths_username: str = None, ths_password: str = None):
        """初始化采集器
        
        Args:
            ths_username: 同花顺账号
            ths_password: 同花顺密码
        """
        self.engine = create_db_engine()
        self.login_status = False
        self.ths_username = ths_username or os.environ.get('THS_USERNAME', '')
        self.ths_password = ths_password or os.environ.get('THS_PASSWORD', '')
        
        # 加载交易日历
        self.trading_days = load_trading_days()
        
    def connect(self) -> bool:
        """连接同花顺数据接口"""
        if self.ths_username and self.ths_password:
            self.login_status = connect_ths(self.ths_username, self.ths_password)
        else:
            print("警告: 未配置同花顺账号密码")
            self.login_status = False
        return self.login_status
    
    def disconnect(self):
        """断开同花顺数据接口"""
        if self.login_status:
            disconnect_ths()
            self.login_status = False
    
    def check_existing_data(self, bond_codes: List[str], start_date: str, end_date: str) -> Dict[str, List[str]]:
        """检查数据库中已存在的数据
        
        Returns:
            Dict[str, List[str]]: 键为转债代码，值为该转债已存在数据的日期列表
        """
        placeholders = ','.join(['%s'] * len(bond_codes))
        query = f"""
            SELECT trade_code, DATE_FORMAT(dt, '%%Y-%%m-%%d') as dt
            FROM marketinfo_equity1
            WHERE trade_code IN ({placeholders})
            AND dt BETWEEN %s AND %s
        """
        
        params = tuple(bond_codes) + (start_date, end_date)
        
        with self.engine.connect() as conn:
            df = pd.read_sql(query, conn, params=params)
        
        existing_data = {}
        for code in bond_codes:
            code_data = df[df['trade_code'] == code]
            existing_data[code] = code_data['dt'].tolist()
        
        return existing_data
    
    def save_to_database(self, df: pd.DataFrame, table_name: str = 'marketinfo_equity1'):
        """保存数据到数据库"""
        df = clean_data(df)
        
        update_columns = [col for col in df.columns if col not in ['dt', 'trade_code']]
        update_stmt = ', '.join([f"{col} = VALUES({col})" for col in update_columns])
        
        # 构建数据字典
        data_dicts = []
        for _, row in df.iterrows():
            row_dict = {}
            for col in df.columns:
                value = row[col]
                if pd.isna(value) or value in [np.inf, -np.inf]:
                    row_dict[col] = None
                else:
                    row_dict[col] = value
            data_dicts.append(row_dict)
        
        # 构建INSERT语句
        columns = list(df.columns)
        placeholders = [f":{col}" for col in columns]
        insert_stmt = f"""
            INSERT INTO {table_name} ({', '.join(columns)})
            VALUES ({', '.join(placeholders)})
            ON DUPLICATE KEY UPDATE {update_stmt}
        """
        
        with self.engine.begin() as conn:
            conn.execute(text(insert_stmt), data_dicts)
    
    def collect_historical_data(self, start_date: str, end_date: str):
        """采集历史数据"""
        # 获取可转债代码列表
        bond_df = get_bond_codes(self.engine)
        total_codes = len(bond_df)
        print(f"共获取到 {total_codes} 个可转债代码")
        
        # 过滤交易日
        trading_days_range = [day for day in self.trading_days 
                             if start_date <= day <= end_date]
        trading_days_range = sorted(trading_days_range)
        print(f"日期范围内共有 {len(trading_days_range)} 个交易日")
        
        # 批量处理
        for i in range(0, len(bond_df), BATCH_SIZE):
            batch_df = bond_df.iloc[i:i + BATCH_SIZE]
            batch_num = i // BATCH_SIZE + 1
            total_batches = (total_codes + BATCH_SIZE - 1) // BATCH_SIZE
            
            print(f"开始处理第 {batch_num}/{total_batches} 批可转债")
            
            batch_codes = batch_df['trade_code'].tolist()
            existing_data = self.check_existing_data(batch_codes, start_date, end_date)
            
            codes_to_update = {}
            for _, row in batch_df.iterrows():
                code = row['trade_code']
                delist_date = pd.to_datetime(row['delist_date']).strftime('%Y-%m-%d') if pd.notna(row['delist_date']) else None
                list_date = pd.to_datetime(row['list_date']).strftime('%Y-%m-%d') if pd.notna(row['list_date']) else None
                existing_dates = set(existing_data.get(code, []))
                
                code_start_date = max(list_date, start_date) if list_date else start_date
                code_end_date = min(end_date, delist_date) if delist_date else end_date
                
                dates_to_fetch = [
                    day for day in trading_days_range
                    if code_start_date <= day <= code_end_date
                    and day not in existing_dates
                ]
                
                if dates_to_fetch:
                    codes_to_update[code] = {
                        'start_date': min(dates_to_fetch),
                        'end_date': max(dates_to_fetch),
                        'missing_dates': set(dates_to_fetch),
                        'delist_date': delist_date
                    }
            
            if not codes_to_update:
                print(f"批次 {batch_num}: 所有数据都已存在，跳过")
                continue
            
            print(f"批次 {batch_num}: 需要更新 {len(codes_to_update)} 个代码")
            
            for idx, (code, info) in enumerate(codes_to_update.items(), 1):
                print(f"处理代码 {code} ({idx}/{len(codes_to_update)})")
                
                df = collect_bond_data([code], info['start_date'], info['end_date'])
                
                if df is not None and not df.empty:
                    for date, group in df.groupby('dt'):
                        date_str = pd.to_datetime(date).strftime('%Y-%m-%d')
                        if date_str not in info['missing_dates']:
                            continue
                        if info['delist_date'] and date_str >= info['delist_date']:
                            continue
                        self.save_to_database(group)
                    print(f"代码 {code} 数据保存成功")
                else:
                    print(f"代码 {code} 未获取到数据")
                
                time.sleep(REQUEST_DELAY)

print("BondDataCollector 类定义完成")

### 5.2 辅助函数

In [ ]:
def setup_logger(name: str, log_dir: str = 'logs') -> logging.Logger:
    """设置日志器
    
    Args:
        name: 日志器名称
        log_dir: 日志目录
        
    Returns:
        logging.Logger: 配置好的日志器
    """
    os.makedirs(log_dir, exist_ok=True)
    
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    
    log_file = os.path.join(log_dir, f'{name}_{datetime.now().strftime("%Y%m%d")}.log')
    fh = logging.FileHandler(log_file, encoding='utf-8')
    fh.setLevel(logging.INFO)
    
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    
    logger.addHandler(fh)
    logger.addHandler(ch)
    
    return logger

print("辅助函数定义完成")

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
# 创建采集器实例 (需要配置环境变量 THS_USERNAME 和 THS_PASSWORD)
collector = BondDataCollector()

# 连接同花顺数据接口
if collector.connect():
    print("连接成功，可以开始采集数据")
    
    # 设置采集日期范围
    start_date = '2025-01-01'
    end_date = '2025-02-15'
    
    # 开始采集数据 (取消注释以执行)
    # collector.collect_historical_data(start_date, end_date)
    
    # 断开连接
    collector.disconnect()
else:
    print("连接失败，请检查账号配置")

### 6.2 结果验证

In [ ]:
# 验证数据库中的数据
def verify_data(engine, table_name: str = 'marketinfo_equity1', limit: int = 10):
    """验证数据库中的数据"""
    query = f"""
        SELECT * FROM {table_name}
        ORDER BY dt DESC
        LIMIT {limit}
    """
    
    with engine.connect() as conn:
        df = pd.read_sql(query, conn)
    
    print(f"最近 {len(df)} 条数据:")
    display(df)
    
    # 统计信息
    count_query = f"SELECT COUNT(*) as total FROM {table_name}"
    with engine.connect() as conn:
        count_result = pd.read_sql(count_query, conn)
    
    print(f"\n数据表总记录数: {count_result['total'].iloc[0]}")

# 执行验证
verify_data(engine)

## 7. 工具函数（可复用）

In [ ]:
class Database:
    """数据库操作工具类"""
    
    def __init__(self, config: dict):
        """初始化数据库连接
        
        Args:
            config: 数据库配置字典，包含 host, port, user, password, database
        """
        self.config = config
        self.engine = self._create_engine()
        
    def _create_engine(self):
        """创建数据库引擎"""
        conn_str = (
            f"mysql+pymysql://{self.config['user']}:{self.config['password']}"
            f"@{self.config['host']}:{self.config['port']}/{self.config['database']}"
        )
        return create_engine(conn_str)
        
    def execute_query(self, query: str, params: Optional[dict] = None) -> pd.DataFrame:
        """执行查询"""
        with self.engine.connect() as conn:
            return pd.read_sql(text(query), conn, params=params)
            
    def execute_update(self, query: str, params: Optional[dict] = None):
        """执行更新"""
        with self.engine.begin() as conn:
            conn.execute(text(query), params or {})
            
    def save_dataframe(self, df: pd.DataFrame, table_name: str, if_exists: str = 'append'):
        """保存DataFrame到数据库"""
        with self.engine.begin() as conn:
            df.to_sql(table_name, conn, if_exists=if_exists, index=False)

print("Database 工具类定义完成")